In [1]:
import geopandas as gpd
import pandas as pd
import libpysal as lps
from sklearn.neighbors import NearestNeighbors
import numpy as np
import esda

import tensorflow as tf
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Dense, Input, Dropout # type: ignore
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt

In [2]:
# handle pre-processing
def get_numeric_cols(dataset):
    """
    Extract numeric columns from a dataset excluding specific columns.
    @param dataset - The dataset from which to extract numeric columns.
    @return The numeric columns excluding LAT_LONG_COLS.
    """
    # numeric columns except for latitude and longitude
    numeric_cols = dataset.select_dtypes(include=['number']).columns
    numeric_cols = numeric_cols.drop(LAT_LONG_COLS + CATEGORICAL_COLS)
    return numeric_cols

# Calculate organic matter metrics if orgc present
def om_bd_columns(df):
    """
    Calculate organic matter and bulk density columns based on the 'orgc' column in the DataFrame.
    @param df - The DataFrame containing the 'orgc' column
    @return The DataFrame with added columns for organic matter and bulk density
    """
    if 'orgc' in df.columns:
        # Use vectorized operations for better performance
        organic_matter = 1.724 * df['orgc']
        df = df.assign(
            organic_matter=organic_matter,
            bulk_density=1.62 - 0.06 * organic_matter
        )
    return df

# Handle silt and clay columns if present
def silt_plus_clay_columns(df):
    """
    Combine the 'silt' and 'clay' columns in the DataFrame by summing them up and creating a new column 'silt_plus_clay'. Then, drop the 'silt' and 'clay' columns from the DataFrame.
    @param df - The DataFrame containing 'silt' and 'clay' columns
    @return The modified DataFrame with a new column 'silt_plus_clay' and without 'silt' and 'clay' columns.
    """
    if {'silt', 'clay'}.issubset(df.columns):
        df['silt_plus_clay'] = df[['silt', 'clay']].sum(axis=1, skipna=True)
        df.drop(columns=['silt', 'clay'], inplace=True)
    return df


In [ ]:
LAT_LONG_COLS = ['latitude', 'longitude']
CATEGORICAL_COLS = ['landcover']
TARGET_COL = 'orgc'

def preprocess_dataset(dataset):
    # convert date column to datetime 
    dataset['date'] = pd.to_datetime(dataset['date'], format='%Y-%m-%d', errors='coerce')

    # filter columns and rows
    min_valid_values = len(dataset) * 0.3
    dataset = (dataset.dropna(thresh=min_valid_values, axis=1)
                    .dropna(subset=[TARGET_COL])
                    .drop_duplicates())

    # add derived features
    dataset = om_bd_columns(dataset)
    dataset = silt_plus_clay_columns(dataset)

    # handle numeric columns
    numeric_cols = get_numeric_cols(dataset)
    df = dataset[numeric_cols].copy()

    # fill missing numeric values with column means
    df = df.fillna(df.mean())

    # add location and categorical columns
    df = pd.concat([
        df,
        dataset[LAT_LONG_COLS],
        pd.get_dummies(dataset[CATEGORICAL_COLS])
    ], axis=1)

    # replace NaN landcover with 0
    df['landcover'] = df['landcover'].fillna(0)

    # convert to geodataframe
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude))

    # drop latitude and longitude
    gdf = gdf.drop(columns=['latitude', 'longitude'])

    print("-"*30)
    print(f"Data after pre-processing: {gdf.shape}")
    print("-"*30)

    return gdf

# load and preprocess dataset
dataset = pd.read_csv("D:/tierra/data/mexico_combined_data.csv")
gdf = preprocess_dataset(dataset)

gdf.head(10)

In [4]:
#%pip install scipy
#%pip install haversine

In [ ]:
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree
import libpysal as lps
import esda
from haversine import haversine, Unit

def spatial_features(gdf):
    if gdf.empty:
        return gdf

    y = gdf[TARGET_COL]

    # Spatial Lag using Queen Contiguity
    try:
        wq = lps.weights.Queen.from_dataframe(gdf, use_index=False)
        wq.transform = 'r'  # Row-standardized weights
        gdf['spatial_lag'] = lps.weights.lag_spatial(wq, y)
    except Exception as e:
        print(f"Spatial lag computation error: {e}")
        gdf['spatial_lag'] = 0

    # Local Moran's I Calculation
    try:
        # Calculate local Moran's I for each location
        local_moran = esda.Moran_Local(y, wq)
        gdf['moran_i'] = local_moran.Is  # This assigns unique Moran's I value for each location
    except Exception as e:
        print(f"Moran's I computation error: {e}")
        gdf['moran_i'] = 0

    # K-Nearest Neighbors Distance and Target Mean Calculation
    try:
        k = 5
        coords = np.column_stack((gdf.geometry.y, gdf.geometry.x))  # (lat, lon) format
        tree = cKDTree(coords)
        _, indices = tree.query(coords, k=k+1)  # k+1 includes the point itself

        # Compute KNN distances using haversine function
        knn_distances = np.array([
            [haversine(coords[i], coords[idx], unit=Unit.KILOMETERS) for idx in indices[i][1:]]
            for i in range(len(coords))
        ])

        gdf['dist_knn_km'] = np.sum(knn_distances, axis=1)

        # Compute mean target value of KNN
        target_values = y.values
        knn_target_values = target_values[indices[:, 1:]]
        gdf['knn_target_mean'] = knn_target_values.mean(axis=1)
    except Exception as e:
        print(f"KNN computation error: {e}")
        gdf['dist_knn_km'] = 0
        gdf['knn_target_mean'] = 0

    # Finding Hotspots and Distance to Nearest Hotspot
    try:
        hotspots = gdf[gdf['spatial_lag'] > y.mean() + y.std()]
        if not hotspots.empty:
            hotspot_coords = np.column_stack((hotspots.geometry.y, hotspots.geometry.x))  # (lat, lon) format
            hotspot_tree = cKDTree(hotspot_coords)
            dist_to_hotspot, indices = hotspot_tree.query(coords)

            # Compute distance to nearest hotspot using haversine function
            gdf['dist_to_hotspot_km'] = np.array([
                haversine(coords[i], hotspot_coords[indices[i]], unit=Unit.KILOMETERS)
                for i in range(len(coords))
            ])
        else:
            gdf['dist_to_hotspot_km'] = np.inf
    except Exception as e:
        print(f"Hotspot distance computation error: {e}")
        gdf['dist_to_hotspot_km'] = np.inf

    return gdf

# Run function
gdf = spatial_features(gdf)

print("-"*30)
print(f"Dataset with spatial features: {gdf.shape}")
print("-"*30)
# pd show all columns
pd.set_option('display.max_columns', None)

gdf.head(10)

In [ ]:
# plot data distribution of each column
import matplotlib.pyplot as plt
import seaborn as sns

# Set up the figure size
plt.figure(figsize=(18, 18))

# Get numerical columns
numeric_cols = gdf.select_dtypes(include=['float64', 'int64']).columns

# Create subplots for each numeric column
n_cols = 2
n_rows = (len(numeric_cols) + 1) // 2
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.histplot(data=gdf, x=col, kde=True)
    plt.title(f'Distribution of {col}')
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Create a separate boxplot for outlier visualization
plt.figure(figsize=(10, 6))
sns.boxplot(data=gdf[numeric_cols])
plt.xticks(rotation=90)
plt.title('Boxplot of Numerical Features')
plt.tight_layout()
plt.show()

In [ ]:
lag_orgc = gdf['spatial_lag']
y = gdf[TARGET_COL]

b, a = np.polyfit(y, lag_orgc, 1)
f, ax = plt.subplots(1, figsize=(9, 9))

plt.plot(y, lag_orgc, '.', color='firebrick')

# dashed vert at mean of the price
plt.vlines(y.mean(), lag_orgc.min(), lag_orgc.max(), linestyle='--')
# dashed horizontal at mean of lagged price
plt.hlines(lag_orgc.mean(), y.min(), y.max(), linestyle='--')

# red line of best fit using global I as slope
plt.plot(y, a + b*y, 'r')
plt.title('Moran Scatterplot')
plt.ylabel('Spatial Lag of Organic Carbon Median')
plt.xlabel('Organic Carbon Median')
plt.show()

In [8]:
def scale_features(gdf, test_size=0.2, random_state=42):
    # deep learning for organic carbon prediction
    X = gdf.drop(columns=['geometry', TARGET_COL])
    y = gdf[TARGET_COL]

    # split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

    # scale features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test, scaler

def build_model(X_train):
    # build the model
    model = Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(64, activation='relu'), # first hidden layer with 64 neurons
        Dropout(0.2), # add dropout to reduce overfitting
        Dense(32, activation='relu'), # second hidden layer with 32 neurons
        Dropout(0.2), # add dropout to reduce overfitting
        Dense(1) # output layer with 1 neuron
    ])

    # compile the model
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    return model

# scale features
X_train, X_test, y_train, y_test, scalar = scale_features(gdf)
# build the model
model = build_model(X_train)

In [ ]:
def train_model(model, X_train, y_train, X_test, y_test, epochs=100, batch_size=32):
    # train the model
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_data=(X_test, y_test), verbose=0)
    return history

# train the model
history = train_model(model, X_train, y_train, X_test, y_test)

# plot the training history
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('Training History Organic Carbon')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_model(model, X_test, y_test):
    """
    Evaluate deep learning model performance using multiple metrics.
    Returns dictionary of metrics and predicted values.
    """
    # Get model predictions
    y_pred = model.predict(X_test, verbose=0).flatten()
    
    # Calculate metrics using sklearn functions for better efficiency
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Get model loss
    loss = model.evaluate(X_test, y_test, verbose=0)[0]
    
    # Store metrics in dictionary
    metrics = {
        'loss': loss,
        'mae': mae, 
        'mse': mse,
        'rmse': rmse,
        'r2': r2
    }
    
    # Print formatted results
    print("\nDeep Learning Model Evaluation:")
    print("-" * 30)
    print(f"Mean Squared Error:     {metrics['mse']:.4f}")
    print(f"Root Mean Squared Error: {metrics['rmse']:.4f}")
    print(f"Mean Absolute Error:    {metrics['mae']:.4f}") 
    print(f"R² Score:              {metrics['r2']:.4f}")
    print("-" * 30)
    
    return metrics, y_pred

# Evaluate model and get metrics
metrics, y_pred = evaluate_model(model, X_test, y_test)

In [ ]:
# plot the predictions vs the actual values
def plot_predictions(y_test, y_pred, metrics, country_name):
    plt.figure(figsize=(10, 6))
    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r:', label='Perfect Prediction')
    plt.xlabel('Actual Organic Carbon (g/Kg)')
    plt.ylabel('Predicted Organic Carbon (g/Kg)')
    plt.title(f'Actual vs Predicted Organic Carbon ({country_name}) \n MSE: {metrics['mse']:.4f}, MAE: {metrics['mae']:.4f}, R2: {metrics['r2']:.4f}')
    plt.legend()
    plt.show()

# plot the predictions vs the actual values
plot_predictions(y_test, y_pred, metrics, 'Mexico')

In [ ]:
import joblib
model_path = "D:/tierra/models/orgc_dl_model.pkl"
scalar_path = "D:/tierra/models/orgc_dl_scalar.pkl"

# Save the model
joblib.dump(model, model_path)
# save the scalar
joblib.dump(scalar, scalar_path)
print("Model saved successfully")

# save the preprocessed data
gdf.to_csv("D:/tierra/datasets/dl_mexico_dataset.csv")

# Testing and Evaluation of the model for different countries


In [13]:
# load the model and sclar
MODEL = joblib.load(model_path)
SCALAR = joblib.load(scalar_path)

In [18]:
def load_test_dataset(dataset, mex_dataset):
    # Get feature columns and ensure they exist in UK data
    feature_columns = mex_dataset.columns
    for col in feature_columns:
        if col not in dataset.columns:
            dataset[col] = 0
    
    # preprocess the dataset
    dataset['date'] = pd.to_datetime(dataset['date'], format='%Y-%m-%d', errors='coerce')

    # filter columns and rows
    dataset = (dataset.dropna(subset=[TARGET_COL])
                    .drop_duplicates())

    # add derived features
    dataset = om_bd_columns(dataset)
    dataset = silt_plus_clay_columns(dataset)

    # handle numeric columns
    df = dataset[feature_columns].copy()
    
    # fill missing numeric values with zero
    df = df.fillna(0)

    # add location and categorical columns
    df = pd.concat([
        df,
        dataset[LAT_LONG_COLS]
    ], axis=1)

    # replace NaN landcover with 0
    df['landcover'] = df['landcover'].fillna(0)

    # convert to geodataframe
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude))

    # drop latitude and longitude
    gdf = gdf.drop(columns=['latitude', 'longitude'])

    print("-"*30)
    print(f"Feature columns: {feature_columns}")
    print(f"Dataset columns: {gdf.columns}")
    print(f"Data after pre-processing: {gdf.shape}")
    print("-"*30)

    # add spatial features
    gdf = spatial_features(gdf)

    return gdf

def test_model(test_gdf, model, scalar, country_name):
    # drop columns
    test_features = test_gdf.drop(columns=['geometry', TARGET_COL])
    # Scale features
    test_gdf_scaled = scalar.transform(test_features)
    # split into features and target
    X_test = test_gdf_scaled
    y_test = test_gdf[TARGET_COL]
    # evaluate the model
    metrics, y_pred = evaluate_model(model, X_test, y_test)
    # plot the predictions vs the actual values
    plot_predictions(y_test, y_pred, metrics, country_name)

## UK dataset

In [ ]:
# United Kingdom dataset
# load and preprocess dataset
uk_dataset = pd.read_csv("D:/tierra/data/United Kingdom_wosis_merged_all.csv")

uk_gdf = load_test_dataset(uk_dataset, gdf)

test_model(uk_gdf, MODEL, SCALAR, "United Kingdom")

# save dataset
uk_gdf.to_csv("D:/tierra/datasets/dl_uk_dataset.csv")

uk_gdf.head(10)

In [50]:
# Argentina dataset
# load and preprocess dataset
arg_dataset = pd.read_csv("D:/tierra/data/Argentina_wosis_merged.csv")

arg_gdf = load_test_dataset(arg_dataset, gdf)

test_model(arg_gdf, MODEL, SCALAR, "India")

# save dataset
arg_gdf.to_csv("D:/tierra/datasets/arg_dataset.csv")

arg_gdf.head(10)

In [ ]:
# India dataset
# load and preprocess dataset
ind_dataset = pd.read_csv("D:/tierra/outputs/India_wosis_merged.csv")

ind_gdf = load_test_dataset(ind_dataset, gdf)

test_model(ind_gdf, MODEL, SCALAR, "India")

# save dataset
ind_gdf.to_csv("D:/tierra/datasets/dl_ind_dataset.csv")

ind_gdf.head(10)